# Finite-Rank Linear System and Residual Gram Matrices

This notebook evaluates a finite-dimensional linear system associated with the linearized operator around the self-similar profile `UP`.

For each basis function `UP_phi_lst[i]`, we consider the linear equation

$$
\mathcal{L}(\phi_i) = \mathrm{rhs}_i,
$$

where

- `UP_phi_lst[i]` represents the left-hand side input $\phi_i$,
- `rhs_lst[i]` represents the right-hand side $\mathrm{rhs}_i$,
- $\mathcal{L}$ is the linear operator implemented in this notebook.

---

## Residual (err_lst)

For each $i$, the residual is defined as

$$
\mathrm{err}_i = \mathrm{rhs}_i - \mathcal{L}(\phi_i).
$$

Thus, `err_lst[i]` is the residual of the linear equation corresponding to the basis function $\phi_i$.

---

## Gram matrices GR and GQ

Two $K \times K$ Gram matrices are constructed using inner products.

### Residual Gram matrix

$$
GR_{ij} =
\langle \mathrm{err}_i,\; \mathrm{err}_j \rangle.
$$

This matrix measures the size and interaction of the residuals.

---

### RHS Gram matrix

$$
GQ_{ij} =
\frac{
\langle \mathrm{rhs}_i,\; \mathrm{rhs}_j \rangle
}{
\lambda_i \lambda_j
},
$$

where $\lambda_i$ are the eigenvalues stored in `eig_val`.

This matrix represents the scaled inner-product structure of the right-hand side functions.

---

## Final printed quantities

The notebook prints

$$
\sqrt{\langle GQ, GR \rangle}
\quad \text{and} \quad
\sqrt{\langle GQ, GQ \rangle},
$$

where the matrix inner product is defined as

$$
\langle A, B \rangle =
\sum_{i,j} A_{ij} B_{ij}.
$$

These quantities are used in error analysis.

In [1]:
import Pkg
Pkg.activate("..")
include("../src/NS_Numerics.jl")
using .NS_Numerics

  Activating 

Using dtype = IntervalArithmetic

project at `~/Changhe/julia code/residual_verification`


.Interval{Float64}
Number of threads = 56


In [2]:
# Load the self-similar profile from MAT file (frequency-domain representation)
UP = from_mat_file("../data/UP.mat", 2; domain="frequency")
M0, M1 = size(UP["u3"].freq_func)

# Load the right-hand side tensor.
# Each slice rhs_tensor[:,:,ind,:] corresponds to rhs_i of the linear system:
#     L(phi_i) = rhs_i
rhs_tensor = read_matrix("../data/rhs.mat", ["rhs_even"])
K = size(rhs_tensor, 3)

# Load eigenvalues and eigenvectors associated with the finite-rank basis.
eig_val, eig_vec = read_matrix("../data/eig.mat", ["eig_val_even", "eig_vec_even"])
# Load parameters related to Bessel zeros used in spectral bounds.
Y_paras, Z_paras = load_besselj_zeros("../data/bessel_zeros.mat"; refine= (dtype != Float64))
# Load the matrix function Q.
# This is used in constructing rhs and operator estimates.
Q = load_grad_U("../data/approx_Q.mat");

In [3]:
# Define domain parameters.
R = dt(32.0)
L0, L1 = Pi / dt(2), Pi / dt(2)
# Sobolev regularity order
k = 12
# Scaling factor relating r and β coordinates.
a = UP.scl_fac / R
Beta = atan(inv(a))

# Compute W^{k,∞} bounds of eigenfunctions in β coordinates.
# These bounds are used to rigorously estimate inner products later.
U_bound, _ = lst_to_mat(Ur_bound_lst(Y_paras, Z_paras, k)...)
eig_bound_beta = r_to_beta(eig_fun_bound(U_bound, eig_vec), a);

# Load the W^{k,∞} bound of Q.
Q_Wk∞_mat = read_matrix("../data/Q_bound.mat", ["Q_WkInf"])
Q_Wk∞_mat_beta = r_to_beta(vec_to_mat(Q_Wk∞_mat), a)

# Bound for rhs_i = Q ψ_i using product/chain-rule based estimates.
# Inner products use weight tanβ/cosβ, so this factor must be included 
# when computing W^{k,∞} bounds for rhs.
B  = tan_over_cos_deriv_Linf_bounds(Beta * dt(1.01), k)
rhs_beta = multinomial_sum(k, B, Q_Wk∞_mat_beta, eig_bound_beta; last_flag=false);

In [4]:
# Construct lists of basis functions and right-hand sides.
UP_phi_lst = [] # stores φ_i (basis functions)
rhs_lst = [] # stores rhs_i
rhs_norms_lst = [] # stores W^{k-2,∞} norm bounds of rhs_i

for ind in 1:K
    println("i = $ind / $K")

    # Load basis function φ_i.
    # These are approximate solutions of the eigenvalue problem.
    UP_phi = from_mat_file("../data/phi_even/UP_phi_$(ind-1).mat", 3; domain="frequency")
    push!(UP_phi_lst, UP_phi)

    # Construct rhs_i as a VectorFunc in physical space.
    rhs = VectorFunc(UP.scl_fac, "space"; e1=rhs_tensor[:, :, ind, 1], e2=rhs_tensor[:, :, ind, 2], e3=rhs_tensor[:, :, ind, 3])
    push!(rhs_lst, rhs)

    # Compute rigorous norm bound of rhs_i.
    paras = Linf_to_paras(rhs_beta, k, dt.([0, 410]); ind=1)
    rhs_norms = norm_lst(rhs, "W$(k-2)Inf"; paras=paras, domain=[Beta, L1])
    push!(rhs_norms_lst, rhs_norms)
end

i = 1 / 19
i = 2 / 19
i = 3 / 19
i = 4 / 19
i = 5 / 19
i = 6 / 19
i = 7 / 19
i = 8 / 19
i = 9 / 19
i = 10 / 19
i = 11 / 19
i = 12 / 19
i = 13 / 19
i = 14 / 19
i = 15 / 19
i = 16 / 19
i = 17 / 19
i = 18 / 19
i = 19 / 19


## Decomposition of the Left-Hand-Side Operator

We consider the linear system

$$
\mathcal{L}(\phi_i) = \mathrm{rhs}_i,
$$

where the left-hand-side operator $\mathcal{L}$ consists of two parts:

$$
\mathcal{L}(\phi)
=
\mathcal{L}_{\mathrm{base}}(\phi)
+
\mathcal{C}_{UP}(\phi),
$$

where

- $\mathcal{L}_{\mathrm{base}}(\phi)$ is the UP-independent linear component,
- $\mathcal{C}_{UP}(\phi)$ is the UP-dependent convection/coupling term involving the self-similar profile `UP`.

---

### Base linear part

The matrix constructed as

$$
\texttt{lin\_mat\_phi}
$$

represents the operator

$$
\mathcal{L}_{\mathrm{base}}(\phi).
$$

It is applied using

$$
\mathcal{L}_{\mathrm{base}}(\phi)
=
\texttt{apply(lin\_mat\_phi, UP\_phi)}.
$$

This part does not involve the profile `UP`.

---

### UP-dependent convection terms

The UP-dependent component is constructed separately using

$$
\texttt{mul\_A\_gradB(UP, UP\_phi)}
\quad \text{and} \quad
\texttt{mul\_A\_gradB(UP\_phi, UP)}.
$$

These correspond to convection/coupling terms involving the profile `UP`.

---

### Full operator and residual

The full left-hand side is assembled by combining both parts:

$$
\mathcal{L}(\phi)
=
\mathcal{L}_{\mathrm{base}}(\phi)
+
\mathcal{C}_{UP}(\phi).
$$

The residual is then computed as

$$
\mathrm{err}_i
=
\mathrm{rhs}_i
-
\mathcal{L}(\phi_i).
$$

This residual is used later to construct the Gram matrix

$$
GR_{ij}
=
\langle \mathrm{err}_i,\; \mathrm{err}_j \rangle.
$$

In [6]:
# Construct physical evaluation grid in β and θ coordinates.
# These grid points are used to apply the linear operator and compute residuals.
N0, N1 = 6000, 3000
β_pt = dt.(0:N0) .*(L0/dt.(N0))
θ_pt = dt.(0:N1) .*(L1/dt.(N1))

# Construct matrix representation of the base linear operator.
# This corresponds to L_base(φ) and does NOT include UP-dependent convection terms.
lin_mat_phi = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "linear");

In [6]:
# Compute UP-dependent convection terms.
# These correspond to:
#     mul_A_gradB(UP, φ) + mul_A_gradB(φ, UP),
# and together form the convection component of the operator.
lfs_lst = vec(mul_A_gradB(UP, UP_phi_lst, β_pt, θ_pt)) .+ vec(mul_A_gradB(UP_phi_lst, UP, β_pt, θ_pt))
lfs_Wk∞_norm_lst = []

# Frequency bounds used in norm estimation. See selfsimilar_profile_verification.ipynb for explanation.
freq_lst = [dt(4 * M0 + 10), dt(4 * M1 + 10)]

for ind in 1:K
    println("i = $ind / $K")

    # Construct full left-hand side operator application
    # Note sign convention follows code implementation
    lfs = lfs_lst[ind] - apply(lin_mat_phi, UP_phi_lst[ind])

    # Compute W^{k-2,∞} bound of the LHS
    lfs_Wk∞_norm = norm_lst(lfs, "W$(k-2)Inf"; paras=freq_lst, domain=[L0,L1])
    push!(lfs_Wk∞_norm_lst, lfs_Wk∞_norm)
end

# Combined bound used later in residual inner product estimates
err_norms_lst = lfs_Wk∞_norm_lst .+ rhs_norms_lst;

i = 1 / 19
i = 2 / 19
i = 3 / 19
i = 4 / 19
i = 5 / 19
i = 6 / 19
i = 7 / 19
i = 8 / 19
i = 9 / 19
i = 10 / 19
i = 11 / 19
i = 12 / 19
i = 13 / 19
i = 14 / 19
i = 15 / 19
i = 16 / 19
i = 17 / 19
i = 18 / 19
i = 19 / 19


In [8]:
# Construct gradient operator matrix
D_phy_mat = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "gradient"; weight_flag=false)
I_mat = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "identity");
linf_norm_lst = []
l2_norm_lst = []
UP_freqs = [dt(2 * M0 + 10), dt(2 * M1 + 10)]

for ind in 1:K
    UP_i = UP_phi_lst[ind]
    # Compute ϕ_i and its gradient
    nabla_UP_i_phy = apply(D_phy_mat, UP_i; new_keys=["A11", "A12", "A13", "A21", "A22", "A23", "A31", "A32", "A33"])
    I_UP_phi = apply(I_mat, UP_i)
    # Compute L2 norm of ϕ_i and L∞ norm of gradient
    UP_i_W1_norm = norm_lst(nabla_UP_i_phy, "LInf"; paras=UP_freqs, domain=[L0,L1])
    UP_i_W_norms = norm_lst(I_UP_phi, "W$(k-2)Inf"; paras=UP_freqs, domain=[L0,L1])
    UP_i_L2_norm = norm_lst(I_UP_phi, "L2"; paras=UP_i_W_norms, domain=[L0,L1])

    push!(l2_norm_lst, norm(UP_i_L2_norm))
    push!(linf_norm_lst, norm(UP_i_W1_norm))
end

println("Linf norm: ", norm(linf_norm_lst))
println("L2 norm: ", norm(l2_norm_lst))

Linf norm: [1.32969, 1.32969]_com
L2 norm: [0.359116, 0.359171]_trv


In [8]:
# Free large tensors to reduce memory footprint before heavier steps.
lfs_lst = nothing
rhs_tensor = nothing
GC.gc(true)

In [9]:
# Compute the aggregate L2 magnitude of the bilinear interaction terms ϕ_i ⋅ ∇ϕ_j.
# This is a global diagnostic for the size of nonlinear/self-interaction among basis functions.
UP_i_nabla_UP_j_lst = mul_A_gradB(UP_phi_lst, UP_phi_lst, β_pt, θ_pt)

sum_grad = zero(dtype)
for UPvec in UP_i_nabla_UP_j_lst
    UPvec_W_norms = norm_lst(UPvec, "W$(k)Inf"; paras=freq_lst, domain=[L0,L1])
    sum_grad += norm_lst(UPvec, "L2"; paras=UPvec_W_norms, domain=[L0,L1])^2
end
println("\\sqrt{\\sum_{i,j} || ϕ_i⋅∇ϕ_j ||_{L^2}^2} = ", sqrt(sum_grad))

\sqrt{\sum_{i,j} || ϕ_i⋅∇ϕ_j ||_{L^2}^2} = [0.104888, 0.111743]_trv


In [10]:
# Release intermediate list and trigger garbage collection.
UP_i_nabla_UP_j_lst = nothing
GC.gc(true)

## Residual construction and Gram matrix assembly

We compute the residual associated with the linear system

$$
\mathcal{L}(\phi_i) = \mathrm{rhs}_i,
$$

where the operator $\mathcal{L}$ consists of

- the base linear part applied via `apply(lin_mat_phi, UP_phi_i)`, and  
- the UP-dependent convection terms computed using `mul_A_gradB`.

The right-hand side `rhs` is supported only on the region

$$
\beta \in [0, \mathrm{Beta}].
$$

Therefore, the residual is evaluated in two regions:

- On $\beta \in [0, \mathrm{Beta}]$:
  $$
  \mathrm{err}_i = \mathrm{rhs}_i + (\text{UP-dependent terms}) - \mathcal{L}_{\mathrm{base}}(\phi_i).
  $$

- On $\beta \in [\mathrm{Beta}, L0]$:
  $$
  \mathrm{rhs}_i = 0,
  \quad
  \mathrm{err}_i = (\text{UP-dependent terms}) - \mathcal{L}_{\mathrm{base}}(\phi_i).
  $$

The residual Gram matrix

$$
GR_{ij} = \langle \mathrm{err}_i,\; \mathrm{err}_j \rangle
$$

is assembled by summing the contributions from both regions.

The RHS Gram matrix

$$
GQ_{ij} =
\frac{\langle \mathrm{rhs}_i,\; \mathrm{rhs}_j \rangle}{\lambda_i \lambda_j}
$$

is computed only on the region where `rhs` is supported.

Finally, the quantities

$$
\sqrt{\langle GQ, GR \rangle}
\quad \text{and} \quad
\sqrt{\langle GQ, GQ \rangle}
$$

are evaluated.

In [11]:
# Refine grid on the near-origin region β ∈ [0, Beta] for tighter residual evaluation.
N0 = 12000
N1 = 6000
β_pt = dt.(0:N0) .*(Beta/dt(N0))
θ_pt = dt.(0:N1) .*(L1/dt(N1))

# Base linear part of the operator on this refined grid (UP-independent part).
lin_mat_phi = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "linear");

In [12]:
# Build residuals err_i on β ∈ [0, Beta].
# We first form the UP-dependent convection/coupling contribution:
#   C_UP(ϕ_i) = mul_A_gradB(UP, ϕ_i) + mul_A_gradB(ϕ_i, UP)
# Then we add rhs_i - L_base(ϕ_i) so that err_i matches the full linear system residual
# under the sign convention used in this notebook.
err_lst = vec(mul_A_gradB(UP, UP_phi_lst, β_pt, θ_pt)) .+ vec(mul_A_gradB(UP_phi_lst, UP, β_pt, θ_pt))

for ind in 1:K
    println("i = $ind / $K")
    
    rhs = rhs_lst[ind]
    # rhs was generated on a grid padded by (k-2) layers; trim removes this padding
    # so residual/inner-product estimates are taken on the interior points only.
    trim_border!(rhs, k-2)

    # Base linear part: L_base(ϕ_i)
    lfs = apply(lin_mat_phi, UP_phi_lst[ind])
    # Residual assembly: err_i += rhs_i - L_base(ϕ_i)
    # (err_lst already contains the UP-dependent term)
    add_equal!(err_lst[ind], rhs - lfs)
end

i = 1 / 19
i = 2 / 19
i = 3 / 19
i = 4 / 19
i = 5 / 19
i = 6 / 19
i = 7 / 19
i = 8 / 19
i = 9 / 19
i = 10 / 19
i = 11 / 19
i = 12 / 19
i = 13 / 19
i = 14 / 19
i = 15 / 19
i = 16 / 19
i = 17 / 19
i = 18 / 19
i = 19 / 19


In [13]:
# Assemble Gram matrices on the region where rhs is supported (β ∈ [0, Beta]).
# At this stage, GR contains only the contribution from the region where rhs ≠ 0.
GQ = zeros(dtype, K, K)
GR = zeros(dtype, K, K)

for i in 1:K, j in i:K
    println("i = $i / $K, j = $j / $K")
    # RHS Gram matrix (rhs support region)
    GQ[i, j] = inner_product(rhs_lst[i], rhs_lst[j]; W_norms_lst=[rhs_norms_lst[i], 
        rhs_norms_lst[j]], domain=[Beta, L1]) / (eig_val[j] * eig_val[i])
        
    # Residual Gram matrix (first-region contribution)
    GR[i, j] = inner_product(err_lst[i], err_lst[j]; W_norms_lst=[err_norms_lst[i], 
        err_norms_lst[j]], domain=[Beta, L1])
    
    # Gram matrices are symmetric; fill lower triangle by copying.
    if i != j
        GQ[j, i] = GQ[i, j]
        GR[j, i] = GR[i, j]
    end
end

i = 1 / 19, j = 1 / 19
i = 1 / 19, j = 2 / 19
i = 1 / 19, j = 3 / 19
i = 1 / 19, j = 4 / 19
i = 1 / 19, j = 5 / 19
i = 1 / 19, j = 6 / 19
i = 1 / 19, j = 7 / 19
i = 1 / 19, j = 8 / 19
i = 1 / 19, j = 9 / 19
i = 1 / 19, j = 10 / 19
i = 1 / 19, j = 11 / 19
i = 1 / 19, j = 12 / 19
i = 1 / 19, j = 13 / 19
i = 1 / 19, j = 14 / 19
i = 1 / 19, j = 15 / 19
i = 1 / 19, j = 16 / 19
i = 1 / 19, j = 17 / 19
i = 1 / 19, j = 18 / 19
i = 1 / 19, j = 19 / 19
i = 2 / 19, j = 2 / 19
i = 2 / 19, j = 3 / 19
i = 2 / 19, j = 4 / 19
i = 2 / 19, j = 5 / 19
i = 2 / 19, j = 6 / 19
i = 2 / 19, j = 7 / 19
i = 2 / 19, j = 8 / 19
i = 2 / 19, j = 9 / 19
i = 2 / 19, j = 10 / 19
i = 2 / 19, j = 11 / 19
i = 2 / 19, j = 12 / 19
i = 2 / 19, j = 13 / 19
i = 2 / 19, j = 14 / 19
i = 2 / 19, j = 15 / 19
i = 2 / 19, j = 16 / 19
i = 2 / 19, j = 17 / 19
i = 2 / 19, j = 18 / 19
i = 2 / 19, j = 19 / 19
i = 3 / 19, j = 3 / 19
i = 3 / 19, j = 4 / 19
i = 3 / 19, j = 5 / 19
i = 3 / 19, j = 6 / 19
i = 3 / 19, j = 7 / 19
i = 3 / 19, j 

In [14]:
# Switch to the complementary region β ∈ [Beta, L0],
# where rhs = 0 and only the operator-dependent part contributes to the residual.
N0 = 6000
N1 = 6000
β_pt = dt.(0:N0) .*((L0 - Beta)/dt(N0)) .+ Beta
θ_pt = dt.(0:N1) .*(L1/dt(N1))

# Base linear operator matrix on this region.
lin_mat_phi = get_operator_matrix(UP_phi_lst[1], β_pt, θ_pt, "linear");

In [15]:
# Construct residual on the region where rhs = 0.
# Since rhs vanishes here, the residual reduces to
#     err_i = (UP-dependent terms) - L_base(φ_i)
err_lst = vec(mul_A_gradB(UP, UP_phi_lst, β_pt, θ_pt)) .+ vec(mul_A_gradB(UP_phi_lst, UP, β_pt, θ_pt))

for ind in 1:K
    println("i = $ind / $K")

    # Base linear part
    lfs = apply(lin_mat_phi, UP_phi_lst[ind])
    # Residual = convection − base linear part
    minus_equal!(err_lst[ind], lfs)
end

i = 1 / 19
i = 2 / 19
i = 3 / 19
i = 4 / 19
i = 5 / 19
i = 6 / 19
i = 7 / 19
i = 8 / 19
i = 9 / 19
i = 10 / 19
i = 11 / 19
i = 12 / 19
i = 13 / 19
i = 14 / 19
i = 15 / 19
i = 16 / 19
i = 17 / 19
i = 18 / 19
i = 19 / 19


In [16]:
# Add the residual Gram matrix contribution from the region where rhs = 0.
# This completes the full-domain residual Gram matrix GR.
for i in 1:K, j in i:K
    println("i = $i / $K, j = $j / $K")
    GR[i, j] += inner_product(err_lst[i], err_lst[j]; W_norms_lst=[lfs_Wk∞_norm_lst[i], 
        lfs_Wk∞_norm_lst[j]], domain=[L0-Beta, L1])
        
    # Gram matrices are symmetric; fill lower triangle by copying.
    if i != j
        GR[j, i] = GR[i, j]
    end
end

i = 1 / 19, j = 1 / 19
i = 1 / 19, j = 2 / 19
i = 1 / 19, j = 3 / 19
i = 1 / 19, j = 4 / 19
i = 1 / 19, j = 5 / 19
i = 1 / 19, j = 6 / 19
i = 1 / 19, j = 7 / 19
i = 1 / 19, j = 8 / 19
i = 1 / 19, j = 9 / 19
i = 1 / 19, j = 10 / 19
i = 1 / 19, j = 11 / 19
i = 1 / 19, j = 12 / 19
i = 1 / 19, j = 13 / 19
i = 1 / 19, j = 14 / 19
i = 1 / 19, j = 15 / 19
i = 1 / 19, j = 16 / 19
i = 1 / 19, j = 17 / 19
i = 1 / 19, j = 18 / 19
i = 1 / 19, j = 19 / 19
i = 2 / 19, j = 2 / 19
i = 2 / 19, j = 3 / 19
i = 2 / 19, j = 4 / 19
i = 2 / 19, j = 5 / 19
i = 2 / 19, j = 6 / 19
i = 2 / 19, j = 7 / 19
i = 2 / 19, j = 8 / 19
i = 2 / 19, j = 9 / 19
i = 2 / 19, j = 10 / 19
i = 2 / 19, j = 11 / 19
i = 2 / 19, j = 12 / 19
i = 2 / 19, j = 13 / 19
i = 2 / 19, j = 14 / 19
i = 2 / 19, j = 15 / 19
i = 2 / 19, j = 16 / 19
i = 2 / 19, j = 17 / 19
i = 2 / 19, j = 18 / 19
i = 2 / 19, j = 19 / 19
i = 3 / 19, j = 3 / 19
i = 3 / 19, j = 4 / 19
i = 3 / 19, j = 5 / 19
i = 3 / 19, j = 6 / 19
i = 3 / 19, j = 7 / 19
i = 3 / 19, j 

In [17]:
# Compute Frobenius-type inner products of Gram matrices.
sum_err = sum(GR[i, j] * GQ[i, j] for i in 1:K, j in 1:K)
println("sqrt(<GQ, GR>): ", sqrt(sum_err))

sum_err = sum(GQ[i, j] * GQ[i, j] for i in 1:K, j in 1:K)
println("sqrt(<GQ, GQ>): ", sqrt(sum_err))

sqrt(<GQ, GR>): [0.0, 0.00118812]_trv
sqrt(<GQ, GQ>): [0.0241888, 0.0243409]_trv
